In [15]:
#🧠 STEP 1 — LOAD DATA FROM SQL
#===============================
import pandas as pd
import sqlite3

# Connect DB
conn = sqlite3.connect(r"C:\Users\acer\Downloads\retailiq.db")

# Pull from your VIEW (IMPORTANT)
df = pd.read_sql("SELECT * FROM customer_360", conn)

df.head()


,CustomerID,Frequency,LastPurchaseDate,FirstPurchaseDate,Monetary,AvgOrderValue
0,12346,1,2011-01-18 10:01:00,2011-01-18 10:01:00,77183.60,77183.600000
1,12347,7,2011-12-07 15:52:00,2010-12-07 14:57:00,4310.00,23.681319
2,12348,4,2011-09-25 13:13:00,2010-12-16 19:09:00,1797.24,57.975484
3,12349,1,2011-11-21 09:51:00,2011-11-21 09:51:00,1757.55,24.076027
4,12350,1,2011-02-02 16:01:00,2011-02-02 16:01:00,334.40,19.670588


In [16]:
#🎯 STEP 2 — CALCULATE RECENCY
#==============================
##👉 Recency = Days since last purchase

from datetime import datetime

# Convert date
df['LastPurchaseDate'] = pd.to_datetime(df['LastPurchaseDate'])

# Reference date (latest date in dataset)
reference_date = df['LastPurchaseDate'].max()

# Recency
df['Recency'] = (reference_date - df['LastPurchaseDate']).dt.days


In [17]:
#🔥 STEP 3 — RFM SCORING (CORE LOGIC)
#=====================================
##🟢 Assign Scores (1–5 scale)
# Recency (lower is better)
df['R_Score'] = pd.qcut(df['Recency'], 5, labels=[5,4,3,2,1])

# Frequency (higher is better)
df['F_Score'] = pd.qcut(df['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])

# Monetary (higher is better)
df['M_Score'] = pd.qcut(df['Monetary'], 5, labels=[1,2,3,4,5])
#🔵 Combine RFM
#===============
df['RFM_Score'] = (
    df['R_Score'].astype(str) +
    df['F_Score'].astype(str) +
    df['M_Score'].astype(str)
)

In [18]:
#🧠 STEP 4 — CUSTOMER SEGMENTATION (VERY IMPORTANT)
#===================================================
def segment_customer(row):
    r = int(row['R_Score'])
    f = int(row['F_Score'])
    m = int(row['M_Score'])

    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    elif r >= 3 and f >= 3:
        return "Loyal Customers"
    elif r <= 2 and f >= 3:
        return "At Risk"
    elif r <= 2 and f <= 2:
        return "Hibernating"
    else:
        return "Others"

df['Segment'] = df.apply(segment_customer, axis=1)


In [19]:
#📊 STEP 5 — VALIDATION (MANDATORY)
#===================================
##🔥 Check average spend per segment
validation = df.groupby('Segment').agg({
    'Monetary': 'mean',
    'Frequency': 'mean'
}).sort_values(by='Monetary', ascending=False)

print(validation)

                    Monetary  Frequency
Segment                                
Champions        6051.819178  11.087409
Loyal Customers  1488.955216   3.714715
At Risk          1250.080826   3.404355
Hibernating       487.707579   1.101408
Others            461.599463   1.200000


In [20]:
#📈 STEP 6 — SAVE FINAL OUTPUT
#==============================
df.to_csv("rfm_output.csv", index=False)

##🔥 BONUS (VERY POWERFUL)
###Top Customers
df.sort_values(by='Monetary', ascending=False).head(10)

,CustomerID,Frequency,LastPurchaseDate,FirstPurchaseDate,Monetary,AvgOrderValue,Recency,R_Score,F_Score,M_Score,RFM_Score,Segment
1689,14646,73,2011-12-08 12:12:00,2010-12-20 10:09:00,280206.02,134.973998,1,5,5,5,555,Champions
4201,18102,60,2011-12-09 11:50:00,2010-12-07 16:42:00,259657.30,602.453132,0,5,5,5,555,Champions
3728,17450,46,2011-12-01 13:29:00,2010-12-07 09:23:00,194550.79,577.302047,7,5,5,5,555,Champions
3008,16446,2,2011-12-09 09:15:00,2011-05-18 09:52:00,168472.50,56157.500000,0,5,3,5,535,Loyal Customers
1879,14911,201,2011-12-08 15:54:00,2010-12-01 14:05:00,143825.06,25.343623,0,5,5,5,555,Champions
55,12415,21,2011-11-15 14:22:00,2011-01-06 11:12:00,124914.53,174.950322,23,4,5,5,455,Champions
1333,14156,55,2011-11-30 10:54:00,2010-12-03 11:48:00,117379.63,83.842593,9,5,5,5,555,Champions
3771,17511,31,2011-12-07 10:12:00,2010-12-01 10:19:00,91062.38,94.561142,2,5,5,5,555,Champions
2702,16029,63,2011-11-01 10:27:00,2010-12-01 09:57:00,81024.84,334.813388,38,3,5,5,355,Loyal Customers
0,12346,1,2011-01-18 10:01:00,2011-01-18 10:01:00,77183.60,77183.600000,325,1,1,5,115,Hibernating
